<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/homework/WRIT20833_HW4_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 4 — Topic Modeling & Integration: What Is the Conversation Actually About?

**WRIT 20833 — Intro to Coding in the Humanities**

**Student Name:** _Replace with your name_

**Upload as:** `LASTNAME_HW4.ipynb`

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

---

## What you're doing
You now have three lenses on a text:
- **HW2 — frequency:** *which words* dominate.
- **HW3 — sentiment:** *how people feel* (tone — which, as you found, is not the same as stance).
- **HW4 — topic modeling (new):** *what sub-conversations* the comments cluster into.

This week you let a tool (**Gensim LDA**) group the comments into **topics**, then you **integrate**
all three: which topics are people talking about, and how do they feel about each one? This is the
move your **capstone** is built on.

## Prepare
Review **our own** materials first:
- **HW2** (`split_into_words`, `stopwords`, `Counter`) and **HW3** (pandas, `df.apply`, sentiment).
- **Pandas 01 / 02** — DataFrames, `groupby`, `df.plot()`.
- The **Topic Modeling (Gensim)** code-along (this week) — dictionary, corpus, `LdaModel`.

## Expected: frequent, meaningful `#comments`, plus written reflections
Comment as you work — not every line, but **often, and wherever you make a choice or hit a
question**: say **what you tried** and **what you learned**. A `#comment` belongs next to the code
it explains: a prediction, a choice, a quick answer.

Longer thinking goes in the **reflection cells**: text cells with the question at the top. Double-click one,
write your answer under the line in plain sentences (no `#` needed), then press **Shift+Enter** to
see it formatted. Under this course's **ungrading** approach, your interpretation — especially
*naming and doubting* the topics — matters more than any single number.

## Load your data (two plumbing cells)
**Before you go on, check the `DATA SOURCE:` line the second cell prints.** It should name your
CSV, the **same file you used in HW3**, so all three lenses describe one dataset. If it says "THIS
IS NOT YOUR DATA," go back and upload your file.

1. **Run the first cell.** A **Choose Files** button appears under it. Click it and pick your clean CSV.
2. **Run the second cell.** It installs **two** tools (VADER for sentiment, **Gensim** for topic
   modeling), loads your file, and re-scores sentiment as in HW3. You don't need to read every line.
   Installing gensim can take a minute and may print a lot. That's normal.

After both run, you have: **`df`** (one comment per row, with a `sentiment` column), **`text_column`**,
the **`analyzer`**, `corpora`/`LdaModel` from gensim, and your **`split_into_words`** + **`stopwords`**.

**Colab forgets uploaded files** when your session ends. When you come back, run both cells again.

**No button?** Some browsers block it. Upload with the **Files** icon (a folder) in the left
sidebar instead. Then, in the first cell, put your file's name between the quotes in
`MY_FILE = ""`, like `MY_FILE = "youtube_comments_abc123.csv"`.

In [ ]:
# PLUMBING: click "Choose Files" and pick your clean CSV (the same one you used in HW3)
MY_FILE = ""   # No button? Upload with the Files icon instead, then type the file name between these quotes.

if MY_FILE == "":
    try:
        from google.colab import files
        uploaded = files.upload()      # waits for you to choose a file; returns {file name: contents}
    except ImportError:                # not in Colab: no upload button
        uploaded = {}
    if len(uploaded) > 0:
        MY_FILE = list(uploaded.keys())[0]
        if len(uploaded) > 1:
            print(f"Warning: You chose {len(uploaded)} files; this notebook uses only the first one.")
        print("Uploaded:", MY_FILE)
    else:
        print("No file was chosen. Run this cell again and pick your clean CSV.")

In [ ]:
# PLUMBING: SETUP — run this cell next. (You're not expected to read every line yet — that's fine.)
import re, os, urllib.request
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# Two borrowed tools: VADER (sentiment, from HW3) and Gensim (topic modeling, new this week).
!pip install -q gensim vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
analyzer = SentimentIntensityAnalyzer()

# Same word-splitter + stopwords from HW2/HW3 (we reuse our own tools).
def split_into_words(any_chunk_of_text):
    return re.split(r"\W+", str(any_chunk_of_text).lower())
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll", "amp"]

def load_course_comments():
    """Load the course YouTube-comments corpus into a DataFrame, one line of comment text per row."""
    filename = "ten_commandments/tc_youtube_comments.txt"
    text = None
    for path in (filename, os.path.join("notebooks", "data", filename),
                 os.path.join("data", filename), os.path.join("..", "data", filename)):
        if os.path.exists(path):
            text = open(path, encoding="utf-8").read(); break
    if text is None:
        url = "https://raw.githubusercontent.com/TCU-DCDA/WRIT20833_2026/main/notebooks/data/" + filename
        text = urllib.request.urlopen(url).read().decode("utf-8")
    rows = [line.strip() for line in text.split("\n") if line.strip()]
    return pd.DataFrame({"comment": rows})


# ----- Load YOUR data -----------------------------------------------------------
MY_TEXT_COLUMN = ""   # leave blank to find it automatically ("comment_clean" or "text_clean")

if MY_FILE:
    MY_FILE = MY_FILE.strip()
    # keep_default_na=False keeps a comment that says "NA" as text instead of an empty cell
    df = pd.read_csv(MY_FILE, keep_default_na=False)
    text_column = MY_TEXT_COLUMN
    if text_column == "":
        for candidate in ["comment_clean", "text_clean"]:
            if candidate in df.columns:
                text_column = candidate
                break
    if text_column not in df.columns:
        raise ValueError(f"Can't tell which column holds your text. Your columns are {list(df.columns)}. "
                         "Type the right one between the quotes in MY_TEXT_COLUMN above and run this cell again.")
    df[text_column] = df[text_column].astype(str)
    blank = df[text_column].str.strip() == ""
    df = df[~blank].reset_index(drop=True)
    print("DATA SOURCE:", MY_FILE, f"({blank.sum()} blank rows dropped)")
else:
    df = load_course_comments()
    text_column = "comment"
    print("DATA SOURCE: course corpus (Ten Commandments comments)")
    print()
    print("*" * 70)
    print("WARNING: THIS IS NOT YOUR DATA.")
    print("No file was uploaded, so you are looking at the course's backup comments.")
    print("HW4 asks you to work on YOUR OWN dataset. Run the upload cell above,")
    print("pick your clean CSV, then run this cell again.")
    print("*" * 70)
    print()

# Sentiment carried over from HW3 (you already know this step):
def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]
df["sentiment"] = df[text_column].apply(get_sentiment)

print(f"{len(df)} rows, sentiment-scored. Text column: '{text_column}'.")
print("First one:", df[text_column].iloc[0][:120])
df.head()

## Part A — From comments to topics (5 exercises)

**A1 — Turn each comment into a list of meaningful words.** *(LDA reads a doc as a bag of words.)*

This is the code-along's `preprocess_for_topics` under a new name, and this time you write it: the
same loop you wrote in HW2, inside a function that handles one comment at a time. `.apply()` (from
HW3) then runs it on every comment.

In [ ]:
# A1
def meaningful_words(comment):
    kept_words = []
    # TODO: loop over the words in split_into_words(comment). Append each word that is
    #       non-empty AND not in stopwords to kept_words. (HW2's filter, one comment at a time.)
    return kept_words

# TODO: make the column (A2 and every part after it use it):
#       df["tokens"] = df[text_column].apply(meaningful_words)
# TODO: print df["tokens"].iloc[0]   (the first comment as a word list)

# #comments:
# This is HW2's "meaningful words" step, now kept PER COMMENT instead of one big pile. Why
# does topic modeling need each document kept separate?

**A2 — Build the dictionary and corpus.** *(Gensim's "bag of words" format.)*

In [ ]:
# A2
dictionary = corpora.Dictionary(df["tokens"])   # every unique word gets an id number

# Read this line as a loop: for each comment's word list, make its bag of words.
corpus = [dictionary.doc2bow(comment_words) for comment_words in df["tokens"]]   # each -> [(word_id, count), ...]

print("vocabulary size:", len(dictionary))
# TODO: print corpus[0]  -- the first comment as (word_id, count) pairs

# Then this line translates the id numbers back into words (another list comprehension: read it as a loop).
print("readable:", [(dictionary[word_id], count) for word_id, count in corpus[0]])

# #comments:
# A "bag of words" throws away word ORDER. What meaning is lost when "dog bites man" and
# "man bites dog" become the same bag?

**A3 — An edge case: predict, then run.**

In [ ]:
# A3 — predict, then run
# Some lines in real data are fragments (e.g. "The"). FIRST predict: after your A1 function drops
# stopwords, what bag of words does a fragment produce? THEN run.
fragment_tokens = meaningful_words("The")
print("tokens:", fragment_tokens)
print("bag of words:", dictionary.doc2bow(fragment_tokens))

# #comments:
# My prediction was:

**A3 reflection** *(double-click to edit; write your answer below the line)*

An **empty** bag means the model has nothing to go on for that comment. What should happen to
empty or fragment documents? Which comments in YOUR data might end up empty (emoji-only, a single
"lol", a reply that just tags someone)?

---

_Your answer:_

**A4 — Train the topic model.** *(Gensim does the math; you choose the settings.)*

In [ ]:
# A4
NUM_TOPICS = 4   # a number YOU choose -- we revisit this in C1
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=NUM_TOPICS,
               random_state=42, passes=10)

# TODO: print each topic with lda.print_topics(num_words=6)
#       (each topic = a weighted list of words the model grouped together)

# #comments:
# Run this cell again -- do the topics look exactly the same? Why might a model's output shift?

**A5 — Read and *name* the topics.** *(The model gives numbers; a human gives meaning.)*

Look at your A4 output. In the markdown cell below, **give each topic a short human label**
(e.g. "Topic 0 = complaints about the ending") and note any topic that seems vague or mixed. There is
no single right answer — that's the point.

**Reflection.** _Double-click this cell and replace this line: list Topic 0…N and your label + one sentence of reasoning for each._

### A note: a topic is a *pattern*, not a fact — and you didn't build the math
Like VADER in HW3, **LDA is a borrowed tool** — you `pip install` gensim; nobody in this course
writes the statistics from scratch. But LDA differs from almost everything else you've run: it is
**stochastic**. Run it twice and the topics can come out in a different order, or shift, because it
starts from randomness (we set `random_state` to pin it down, but a different computer or gensim
version can still differ).

So a "topic" is **not a fact the data contains** — it's a *pattern the model proposes*, and a
**human decides** whether it means anything and what to call it (that's A5). Two analysts can label
the same topics differently. Hold your topics loosely. *(Day 9 again: judging a tool's output, not
just running it.)*

## Part B — Integration: topics meet sentiment (4 exercises)

**B1 — Assign each comment its dominant topic.**

In [ ]:
# B1
def dominant_topic(bag_of_words):
    if not bag_of_words:                    # empty/fragment doc -> no real topic
        return -1
    topic_probs = lda.get_document_topics(bag_of_words)   # [(topic_number, probability), ...]
    # max(..., key=...) finds the pair with the highest probability; [0] keeps its topic number.
    return max(topic_probs, key=lambda pair: pair[1])[0]

topics = []
# TODO: write a for loop over corpus. For each bag_of_words, append dominant_topic(bag_of_words) to topics.

# One topic per comment (B2 and B3 use this column). If this line says "Length of values (0) does
# not match", your loop above hasn't appended anything to topics yet.
df["topic"] = topics
# TODO: print df[[text_column, "topic"]].head()

# #comments:
# Each comment is now filed under ONE topic. What is lost by forcing a comment that touches
# two themes into a single bin?

**B2 — How many comments per topic?** *(Plus a quick chart.)*

In [ ]:
# B2
# TODO: print df["topic"].value_counts().sort_index()
# TODO: make a bar chart:
#       df["topic"].value_counts().sort_index().plot(kind="bar", title="Comments per topic")

# #comments:
# Are the topics evenly sized, or does one dominate? (A "-1" bar = empty/fragment comments.)

**B3 — The integration: how does each topic *feel*?** *(HW3 sentiment, grouped by HW4 topic.)*

In [ ]:
# B3
# TODO: print the AVERAGE sentiment for each topic:
#       df.groupby("topic")["sentiment"].mean().round(3)

**B3 reflection** *(double-click to edit; write your answer below the line)*

This is the whole course in one line: frequency found the words, sentiment found the feeling,
topics found the sub-conversations, and now you can say **which** sub-conversation has the lowest
or highest average VADER score. An average can hide a split, so also count each topic (B2) and read
a few of its comments. Is the lowest-scoring topic angry, or just full of negative words?

---

_Your answer:_

**B4 — Interpretation (write 5–7 sentences below).**

Bring all three lenses together for **one claim about this conversation.** Use specifics: name a
topic and its label (A5), its average sentiment (B3), and a word that "won" by frequency (your top
word from Count Your Data, or HW3's C1). What is this conversation *actually about*, and how does each part of it feel? Then the honest
part: name **one place the tools mislead** — a vague topic, a sentiment score you distrust, or an
empty/fragment document — and say how you'd check it by hand. *(This is exactly your capstone in
miniature.)* For the check, [Checking the Machine](https://github.com/TCU-DCDA/WRIT20833_2026/blob/main/materials/Checking_the_Machine.md) gives you a routine: sample a few
documents per topic, label them yourself first, compare, then keep, narrow, or decline your claim.

**Reflection.** _Double-click this cell and replace this line with your integrated interpretation._

## Part C — Going deeper / capstone bridge (2 exercises)

**C1 — Change the number of topics.** *(The output depends on a number you picked.)*

In [ ]:
# C1
# TODO: train a model with NUM_TOPICS = 2, print its topics.
# TODO: train another with NUM_TOPICS = 6, print its topics.
# (Copy the A4 pattern; just change num_topics. Keep random_state=42.)

**C1 reflection** *(double-click to edit; write your answer below the line)*

With 2 topics the story is coarse; with 6 it fragments. There is no "true" number: **you**
author it. How would you defend the number you'd choose for your capstone?

---

_Your answer:_

**C2 — Capstone bridge: revisit your proposal.**

You turned in your capstone proposal on Day 18 — before you'd run all three lenses on data yourself.
Now you have. In the markdown cell below, write 4–5 sentences looking back at that proposal: which of
the three lenses (frequency / sentiment / topics) best fits your question *now*, what you'd change
(dataset, question, or method), and one limitation you'll be honest about. The proposal isn't a
contract; changing it because of what you learned here is the point. Say what surprised you about
your own data.

**Reflection.** _Double-click this cell and replace this line with your look back at your proposal._

## Weekly Experiments (your own work!)
After the required exercises, create **2–3 small experiments** of your own. Make them original.

Ideas:
- Re-run A4 a few times with `random_state` removed — watch the topics move.
- Print the 3 most *positive* and 3 most *negative* comments **within one topic**.
- Add a few custom stopwords and see whether the topics get cleaner.

It's fine if an experiment doesn't work at first — **documenting what went wrong and what you
learned is worth just as much as working code.**

In [ ]:
# Experiment 1
# (#comments in your code as you go)

**Experiment 1 reflection** *(double-click to edit; write your answer below the line)*

What were you trying? What happened? What did you learn?

---

_Your answer:_

In [ ]:
# Experiment 2
# (#comments in your code as you go)

**Experiment 2 reflection** *(double-click to edit; write your answer below the line)*

What were you trying? What happened? What did you learn?

---

_Your answer:_

In [ ]:
# Experiment 3
# (#comments in your code as you go)

**Experiment 3 reflection** *(double-click to edit; write your answer below the line)*

What were you trying? What happened? What did you learn?

---

_Your answer:_

## Submit
- [ ] My name is at the top.
- [ ] I uploaded **my own** clean CSV (the same one as HW3), and the `DATA SOURCE:` line names my file (not "THIS IS NOT YOUR DATA"). If I couldn't get my file to load, I wrote a `#comment` in the setup cell saying what went wrong. I ran every cell. Where something still breaks, I left a `#comment` about what I tried and what I learned — errors are part of the work here, not something to hide or delete.
- [ ] I completed **Part A (5, incl. naming the topics)**, **Part B (4, incl. the B4 write-up)**, **Part C (2)**, and **2–3 experiments**.
- [ ] I used `#comments` frequently and meaningfully — showing my thinking wherever I made a choice or hit a question (not every line, but throughout).
- [ ] I answered every reflection in its text cell, under the line.
- [ ] I **named and doubted** my topics honestly (engagement matters more than "right" topics).
- [ ] Saved as `LASTNAME_HW4.ipynb` and uploaded to its folder in TCU Online → Assignments.